In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    _p = _root / "paths.py"
    if _p.exists() and "SHARED_ROOT" in _p.read_text(encoding="utf-8", errors="ignore"):
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find After_PT_Removal/shared/paths.py — set Jupyter cwd to After_PT_Removal, shared/, or shared/notebooks/."
    )
import paths


In [1]:
import pandas as pd

# Load the model predictions
output_path = paths.DATA / "Centaur_Lab_Second_Round.csv"
df = pd.read_csv(output_path)
df

,Origin,q1,q2,q3,q4,q5,q6,q7,q8,q9,...,ID_corr,sentence_number_corr,answer_corr,data_source_corr,REMOVED_Sentences,sentence_number_df3,step1_excerpts,question_options,Filtered_Sentences,New_Sentences
0,ID0002,1,0.833333333,REMOVED,1,REMOVED,0.833333333,REMOVED,REMOVED,REMOVED,...,ID0002,11,D,jama,6,11,1. A woman in her 60s with a history of hyperl...,What Would You Do Next?\n\nA: Perform left upp...,"['3. On examination, corrected visual acuity w...",['1. A woman in her 60s with a history of hype...
1,ID0003,1,1,REMOVED,1,1,1,REMOVED,NaN,NaN,...,ID0003,7,G,medxpert,2,7,1. A 20-year-old woman comes to the primary ca...,Which of the following components is essential...,"['3. Her medical history is unremarkable, and ...",['1. A 20-year-old woman comes to the primary ...
2,ID0007,REMOVED,1,1,1,1,REMOVED,1,NaN,NaN,...,ID0007,7,B,medbullets,2,7,1. A 72-year-old man presents to his primary c...,Which of the following diagnostic tests would ...,['1. A 72-year-old man presents to his primary...,['2. He has felt very weak every morning with ...
3,ID0009,REMOVED,1,1,1,1,1,REMOVED,REMOVED,NaN,...,ID0009,8,D,jama,3,8,1. A woman in her 30s presented with multiple ...,What Is Your Diagnosis?\n\nA: Blue rubber bleb...,['1. A woman in her 30s presented with multipl...,['2. The lesions had been present since childh...
4,ID0010,1,REMOVED,REMOVED,REMOVED,REMOVED,REMOVED,1,NaN,NaN,...,ID0010,7,H,medxpert,5,7,1. A 17-year-old high school student accidenta...,What is the proper method for transporting the...,['2. His teacher applied dressings and pressur...,['1. A 17-year-old high school student acciden...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1295,ID1995,1,0.75,REMOVED,REMOVED,0.75,NaN,NaN,NaN,NaN,...,ID1995,5,D,mmlu,2,5,1. A 17-year-old girl is brought to the emerge...,Which of the following types of drugs is the m...,"['3. Her temperature is 37.1°C (98.8°F), pulse...",['1. A 17-year-old girl is brought to the emer...
1296,ID1996,1,REMOVED,1,NaN,NaN,NaN,NaN,NaN,NaN,...,ID1996,3,D,mmlu,1,3,1. A 68-year-old female presents to the emerge...,The most likely etiologic organism is\n\nA. Cl...,['2. Today the patient is nauseated and less r...,['1. A 68-year-old female presents to the emer...
1297,ID1997,1,1,REMOVED,1,REMOVED,REMOVED,REMOVED,REMOVED,REMOVED,...,ID1997,12,B,medxpert,9,12,1. A 27-year-old woman presents with a 4-month...,What is the most appropriate next step in the ...,"['3. She denies nipple discharge.', '5. She is...",['1. A 27-year-old woman presents with a 4-mon...
1298,ID1998,0.5,0.5,0.5,0.5,0.5,0.5,REMOVED,REMOVED,NaN,...,ID1998,8,A,medbullets,2,8,1. A 6-month-old girl is brought to the pediat...,Which of the structures labeled in Figure A wo...,"['7. A physical examination is unremarkable.',...",['1. A 6-month-old girl is brought to the pedi...


# Use Irrelevant + Low Relevance Sentences to make predictions

In [ ]:
<REMOVED_USE_OPENAI_API_KEY_ENV>

In [4]:
import openai

def generate_direct_prediction(context, question):
    """
    Queries GPT-4o with a clinical vignette (context) and a multiple-choice question (with embedded options).
    Returns only the predicted answer in the format: '[Letter]: [Answer Text]' (e.g., 'B: Femoral artery murmur').
    """
    prompt = f"""
You are given some context and a multiple-choice question.

Select the most appropriate answer from the options provided.

{context}

{question}

Provide your response in the following format:\n<answer>Option [letter]</answer>"""

    try:
        client = openai.OpenAI()

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature = 0
        )

        return response.choices[0].message.content.strip()

    except Exception:
        return "Error"


In [5]:
# Apply the prediction function to the first 5 rows of the DataFrame
df.loc[:4, "gpt_direct_prediction"] = df.loc[:4].apply(
    lambda row: generate_direct_prediction(
        row["Filtered_Sentences"], row["question_options"]
    ),
    axis=1
)

# Display the question and GPT-4o prediction for inspection
print(df[["question_options", "gpt_direct_prediction"]].head(5))


                                    question_options  \
0  What Would You Do Next?\n\nA: Perform left upp...   
1  Which of the following components is essential...   
2  Which of the following diagnostic tests would ...   
3  What Is Your Diagnosis?\n\nA: Blue rubber bleb...   
4  What is the proper method for transporting the...   

       gpt_direct_prediction  
0  <answer>Option D</answer>  
1  <answer>Option D</answer>  
2  <answer>Option B</answer>  
3  <answer>Option D</answer>  
4  <answer>Option H</answer>  


In [2]:
import pandas as pd
from tqdm import tqdm
import os

output_path = paths.GPT4O_PREDICTIONS / "gpt4o_predictions_on_trainee_irr_removed.csv"

# If continuing from a previous batch, load the existing file and get already-completed indices
if os.path.exists(output_path):
    df_existing = pd.read_csv(output_path)
    completed_ids = set(df_existing.index)
    print(f"✅ Loaded existing file with {len(completed_ids)} completed rows.")
else:
    df_existing = pd.DataFrame()
    completed_ids = set()

# Collect new results in a list of dicts
new_rows = []

# Iterate with progress bar
for idx, row in tqdm(df.iterrows(), total=len(df)):
    if idx in completed_ids:
        continue  # skip already processed

    pred = generate_direct_prediction(row["Filtered_Sentences"], row["question_options"])
    
    result_row = row.to_dict()
    result_row["gpt_direct_prediction"] = pred
    new_rows.append(result_row)

    # Write out after each row to ensure persistence
    df_batch = pd.DataFrame(new_rows)
    df_combined = pd.concat([df_existing, df_batch], ignore_index=True)
    df_combined.to_csv(output_path, index=False)


✅ Loaded existing file with 1300 completed rows.


100%|██████████| 1300/1300 [00:00<00:00, 18494.66it/s]


In [3]:
import pandas as pd
import re
import numpy as np

# Load the model predictions
output_path = paths.GPT4O_PREDICTIONS / "gpt4o_predictions_on_trainee_irr_removed.csv"
df = pd.read_csv(output_path)

# Extract the predicted letter from the format <answer>Option A</answer>
def extract_letter_from_xml(pred):
    if isinstance(pred, str):
        match = re.search(r"Option\s+([A-J])", pred, re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None

df["gpt_letter"] = df["gpt_direct_prediction"].apply(extract_letter_from_xml)

# # Clean and standardize the ground truth answer
df["answer_letter"] = df["answer_corr"].astype(str).str.strip().str.upper()

# Compare predictions to ground truth
df["gpt_letter_match"] = df.apply(
    lambda row: "Correct" if row["gpt_letter"] == row["answer_letter"] else "Incorrect",
    axis=1
)

# Convert to binary for std calculation
df["gpt_letter_binary"] = df["gpt_letter_match"].map({"Correct": 1, "Incorrect": 0})

# Compute overall accuracy
correct_count = df["gpt_letter_binary"].sum()
total_count = df["gpt_letter_binary"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0
std = df["gpt_letter_binary"].std(ddof=1) if total_count > 1 else float("nan")

print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")
print(f"Letter Match Standard Deviation: {std:.3f}")

# # Per-data source accuracy and std
for source in df["data_source_corr"].unique():
    source_df = df[df["data_source_corr"] == source]
    correct = source_df["gpt_letter_binary"].sum()
    total = source_df["gpt_letter_binary"].notna().sum()
    acc = correct / total if total > 0 else 0
    std = source_df["gpt_letter_binary"].std(ddof=1) if total > 1 else float("nan")
    
    print(f"Data Source: {source}")
    print(f"  Correct Predictions: {correct}")
    print(f"  Total Predictions: {total}")
    print(f"  Accuracy: {acc:.2%}")
    print(f"  Std Dev: {std:.4f}\n")


Letter-Based Correct Predictions: 579
Total Predictions Compared: 1300
Letter Match Accuracy: 44.54%
Letter Match Standard Deviation: 0.497
Data Source: jama
  Correct Predictions: 281
  Total Predictions: 582
  Accuracy: 48.28%
  Std Dev: 0.5001

Data Source: medxpert
  Correct Predictions: 79
  Total Predictions: 318
  Accuracy: 24.84%
  Std Dev: 0.4328

Data Source: medbullets
  Correct Predictions: 95
  Total Predictions: 207
  Accuracy: 45.89%
  Std Dev: 0.4995

Data Source: mmlu
  Correct Predictions: 124
  Total Predictions: 193
  Accuracy: 64.25%
  Std Dev: 0.4805



In [ ]:
import pandas as pd

# Load the model predictions
output_path = paths.DATA / "Centaur_Lab_Second_Round.csv"
df = pd.read_csv(output_path)
df

,Origin,q1,q2,q3,q4,q5,q6,q7,q8,q9,...,ID_corr,sentence_number_corr,answer_corr,data_source_corr,REMOVED_Sentences,sentence_number_df3,step1_excerpts,question_options,Filtered_Sentences,New_Sentences
0,ID0002,1,0.833333333,REMOVED,1,REMOVED,0.833333333,REMOVED,REMOVED,REMOVED,...,ID0002,11,D,jama,6,11,1. A woman in her 60s with a history of hyperl...,What Would You Do Next?\n\nA: Perform left upp...,"['3. On examination, corrected visual acuity w...",['1. A woman in her 60s with a history of hype...
1,ID0003,1,1,REMOVED,1,1,1,REMOVED,NaN,NaN,...,ID0003,7,G,medxpert,2,7,1. A 20-year-old woman comes to the primary ca...,Which of the following components is essential...,"['3. Her medical history is unremarkable, and ...",['1. A 20-year-old woman comes to the primary ...
2,ID0007,REMOVED,1,1,1,1,REMOVED,1,NaN,NaN,...,ID0007,7,B,medbullets,2,7,1. A 72-year-old man presents to his primary c...,Which of the following diagnostic tests would ...,['1. A 72-year-old man presents to his primary...,['2. He has felt very weak every morning with ...
3,ID0009,REMOVED,1,1,1,1,1,REMOVED,REMOVED,NaN,...,ID0009,8,D,jama,3,8,1. A woman in her 30s presented with multiple ...,What Is Your Diagnosis?\n\nA: Blue rubber bleb...,['1. A woman in her 30s presented with multipl...,['2. The lesions had been present since childh...
4,ID0010,1,REMOVED,REMOVED,REMOVED,REMOVED,REMOVED,1,NaN,NaN,...,ID0010,7,H,medxpert,5,7,1. A 17-year-old high school student accidenta...,What is the proper method for transporting the...,['2. His teacher applied dressings and pressur...,['1. A 17-year-old high school student acciden...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1295,ID1995,1,0.75,REMOVED,REMOVED,0.75,NaN,NaN,NaN,NaN,...,ID1995,5,D,mmlu,2,5,1. A 17-year-old girl is brought to the emerge...,Which of the following types of drugs is the m...,"['3. Her temperature is 37.1°C (98.8°F), pulse...",['1. A 17-year-old girl is brought to the emer...
1296,ID1996,1,REMOVED,1,NaN,NaN,NaN,NaN,NaN,NaN,...,ID1996,3,D,mmlu,1,3,1. A 68-year-old female presents to the emerge...,The most likely etiologic organism is\n\nA. Cl...,['2. Today the patient is nauseated and less r...,['1. A 68-year-old female presents to the emer...
1297,ID1997,1,1,REMOVED,1,REMOVED,REMOVED,REMOVED,REMOVED,REMOVED,...,ID1997,12,B,medxpert,9,12,1. A 27-year-old woman presents with a 4-month...,What is the most appropriate next step in the ...,"['3. She denies nipple discharge.', '5. She is...",['1. A 27-year-old woman presents with a 4-mon...
1298,ID1998,0.5,0.5,0.5,0.5,0.5,0.5,REMOVED,REMOVED,NaN,...,ID1998,8,A,medbullets,2,8,1. A 6-month-old girl is brought to the pediat...,Which of the structures labeled in Figure A wo...,"['7. A physical examination is unremarkable.',...",['1. A 6-month-old girl is brought to the pedi...


In [ ]:
import pandas as pd

# Load the model predictions
output_path = paths.DATA / "Centaur_Lab_Second_Round.csv"
df = pd.read_csv(output_path)
df

,Origin,q1,q2,q3,q4,q5,q6,q7,q8,q9,...,ID_corr,sentence_number_corr,answer_corr,data_source_corr,REMOVED_Sentences,sentence_number_df3,step1_excerpts,question_options,Filtered_Sentences,New_Sentences
0,ID0002,1,0.833333333,REMOVED,1,REMOVED,0.833333333,REMOVED,REMOVED,REMOVED,...,ID0002,11,D,jama,6,11,1. A woman in her 60s with a history of hyperl...,What Would You Do Next?\n\nA: Perform left upp...,"['3. On examination, corrected visual acuity w...",['1. A woman in her 60s with a history of hype...
1,ID0003,1,1,REMOVED,1,1,1,REMOVED,NaN,NaN,...,ID0003,7,G,medxpert,2,7,1. A 20-year-old woman comes to the primary ca...,Which of the following components is essential...,"['3. Her medical history is unremarkable, and ...",['1. A 20-year-old woman comes to the primary ...
2,ID0007,REMOVED,1,1,1,1,REMOVED,1,NaN,NaN,...,ID0007,7,B,medbullets,2,7,1. A 72-year-old man presents to his primary c...,Which of the following diagnostic tests would ...,['1. A 72-year-old man presents to his primary...,['2. He has felt very weak every morning with ...
3,ID0009,REMOVED,1,1,1,1,1,REMOVED,REMOVED,NaN,...,ID0009,8,D,jama,3,8,1. A woman in her 30s presented with multiple ...,What Is Your Diagnosis?\n\nA: Blue rubber bleb...,['1. A woman in her 30s presented with multipl...,['2. The lesions had been present since childh...
4,ID0010,1,REMOVED,REMOVED,REMOVED,REMOVED,REMOVED,1,NaN,NaN,...,ID0010,7,H,medxpert,5,7,1. A 17-year-old high school student accidenta...,What is the proper method for transporting the...,['2. His teacher applied dressings and pressur...,['1. A 17-year-old high school student acciden...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1295,ID1995,1,0.75,REMOVED,REMOVED,0.75,NaN,NaN,NaN,NaN,...,ID1995,5,D,mmlu,2,5,1. A 17-year-old girl is brought to the emerge...,Which of the following types of drugs is the m...,"['3. Her temperature is 37.1°C (98.8°F), pulse...",['1. A 17-year-old girl is brought to the emer...
1296,ID1996,1,REMOVED,1,NaN,NaN,NaN,NaN,NaN,NaN,...,ID1996,3,D,mmlu,1,3,1. A 68-year-old female presents to the emerge...,The most likely etiologic organism is\n\nA. Cl...,['2. Today the patient is nauseated and less r...,['1. A 68-year-old female presents to the emer...
1297,ID1997,1,1,REMOVED,1,REMOVED,REMOVED,REMOVED,REMOVED,REMOVED,...,ID1997,12,B,medxpert,9,12,1. A 27-year-old woman presents with a 4-month...,What is the most appropriate next step in the ...,"['3. She denies nipple discharge.', '5. She is...",['1. A 27-year-old woman presents with a 4-mon...
1298,ID1998,0.5,0.5,0.5,0.5,0.5,0.5,REMOVED,REMOVED,NaN,...,ID1998,8,A,medbullets,2,8,1. A 6-month-old girl is brought to the pediat...,Which of the structures labeled in Figure A wo...,"['7. A physical examination is unremarkable.',...",['1. A 6-month-old girl is brought to the pedi...


# Use High Relevance Sentences to make predictions

In [2]:
import pandas as pd, numpy as np, re, ast
from pandas.api.types import is_scalar

# ------------------------------------------------------------------
# 1.  regex to strip a leading number + punctuation
# ------------------------------------------------------------------
_strip_num = re.compile(r'^\s*\d+\s*[\.\)\-:]?\s*').sub

# ------------------------------------------------------------------
# 2.  cleaner for each cell in New_Sentences
# ------------------------------------------------------------------
def tidy_sentence_list(cell) -> str:
    """
    Return a single plain‑text string: sentences joined by one space,
    each stripped of any leading enumeration such as '1. ', '02) ', etc.
    """
    # --- handle missing values safely ---------------------------------------
    if is_scalar(cell) and pd.isna(cell):
        return ""

    # --- ensure we have a Python list of sentence strings -------------------
    if isinstance(cell, str):
        cell = cell.strip()
        if cell.startswith("[") and cell.endswith("]"):
            # looks like "['…', '…']" → turn into real list
            try:
                cell = ast.literal_eval(cell)
            except (ValueError, SyntaxError):
                cell = [cell]                     # fall back: wrap the string
        else:
            cell = [cell]

    elif not isinstance(cell, (list, tuple)):
        # any other non‑list object → make it a single‑item list
        cell = [str(cell)]

    # --- clean each sentence ------------------------------------------------
    cleaned = [_strip_num("", str(s)).strip() for s in cell if str(s).strip()]
    return " ".join(cleaned)

# ------------------------------------------------------------------
# 3.  apply to the column
# ------------------------------------------------------------------
df["New_Sentences"] = df["New_Sentences"].apply(tidy_sentence_list)

# quick check
df["New_Sentences"].head()


0    A woman in her 60s with a history of hyperlipi...
1    A 20-year-old woman comes to the primary care ...
2    He has felt very weak every morning with his s...
3    The lesions had been present since childhood, ...
4    A 17-year-old high school student accidentally...
Name: New_Sentences, dtype: object

In [13]:
import openai

def generate_direct_prediction(context, question):
    """
    Queries GPT-4o with a clinical vignette (context) and a multiple-choice question (with embedded options).
    Returns only the predicted answer in the format: '[Letter]: [Answer Text]' (e.g., 'B: Femoral artery murmur').
    """
    prompt = f"""
You are given some context and a multiple-choice question.

Select the most appropriate answer from the options provided.

{context}

{question}

Provide your response in the following format:\n<answer>Option [letter]</answer>"""

    try:
        client = openai.OpenAI()

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature = 0
        )

        return response.choices[0].message.content.strip()

    except Exception:
        return "Error"


In [14]:
# Apply the prediction function to the first 5 rows of the DataFrame
df.loc[:4, "gpt_direct_prediction"] = df.loc[:4].apply(
    lambda row: generate_direct_prediction(
        row["New_Sentences"], row["question_options"]
    ),
    axis=1
)

# Display the question and GPT-4o prediction for inspection
print(df[["question_options", "gpt_direct_prediction"]].head(5))


                                    question_options  \
0  What Would You Do Next?\n\nA: Perform left upp...   
1  Which of the following components is essential...   
2  Which of the following diagnostic tests would ...   
3  What Is Your Diagnosis?\n\nA: Blue rubber bleb...   
4  What is the proper method for transporting the...   

       gpt_direct_prediction  
0  <answer>Option D</answer>  
1  <answer>Option D</answer>  
2  <answer>Option B</answer>  
3  <answer>Option D</answer>  
4  <answer>Option A</answer>  


In [15]:
import pandas as pd
from tqdm import tqdm
import os

output_path = paths.GPT4O_PREDICTIONS / "gpt4o_predictions_on_trainee_removed.csv"

# If continuing from a previous batch, load the existing file and get already-completed indices
if os.path.exists(output_path):
    df_existing = pd.read_csv(output_path)
    completed_ids = set(df_existing.index)
    print(f"✅ Loaded existing file with {len(completed_ids)} completed rows.")
else:
    df_existing = pd.DataFrame()
    completed_ids = set()

# Collect new results in a list of dicts
new_rows = []

# Iterate with progress bar
for idx, row in tqdm(df.iterrows(), total=len(df)):
    if idx in completed_ids:
        continue  # skip already processed

    pred = generate_direct_prediction(row["New_Sentences"], row["question_options"])
    
    result_row = row.to_dict()
    result_row["gpt_direct_prediction"] = pred
    new_rows.append(result_row)

    # Write out after each row to ensure persistence
    df_batch = pd.DataFrame(new_rows)
    df_combined = pd.concat([df_existing, df_batch], ignore_index=True)
    df_combined.to_csv(output_path, index=False)


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1300/1300 [11:02<00:00,  1.96it/s]


In [4]:
import pandas as pd
import re
import numpy as np

# Load the model predictions
output_path = paths.GPT4O_PREDICTIONS / "gpt4o_predictions_on_trainee_removed.csv"
df = pd.read_csv(output_path)

# Extract the predicted letter from the format <answer>Option A</answer>
def extract_letter_from_xml(pred):
    if isinstance(pred, str):
        match = re.search(r"Option\s+([A-J])", pred, re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None

df["gpt_letter"] = df["gpt_direct_prediction"].apply(extract_letter_from_xml)

# # Clean and standardize the ground truth answer
df["answer_letter"] = df["answer_corr"].astype(str).str.strip().str.upper()

# Compare predictions to ground truth
df["gpt_letter_match"] = df.apply(
    lambda row: "Correct" if row["gpt_letter"] == row["answer_letter"] else "Incorrect",
    axis=1
)

# Convert to binary for std calculation
df["gpt_letter_binary"] = df["gpt_letter_match"].map({"Correct": 1, "Incorrect": 0})

# Compute overall accuracy
correct_count = df["gpt_letter_binary"].sum()
total_count = df["gpt_letter_binary"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0
std = source_df["gpt_letter_binary"].std(ddof=1) if total > 1 else float("nan")

print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")
print(f"Letter Match Standard Deviation: {std:.3f}")

# # Per-data source accuracy and std
for source in df["data_source_corr"].unique():
    source_df = df[df["data_source_corr"] == source]
    correct = source_df["gpt_letter_binary"].sum()
    total = source_df["gpt_letter_binary"].notna().sum()
    acc = correct / total if total > 0 else 0
    std = source_df["gpt_letter_binary"].std(ddof=1) if total > 1 else float("nan")
    
    print(f"Data Source: {source}")
    print(f"  Correct Predictions: {correct}")
    print(f"  Total Predictions: {total}")
    print(f"  Accuracy: {acc:.2%}")
    print(f"  Std Dev: {std:.4f}\n")


Letter-Based Correct Predictions: 949
Total Predictions Compared: 1300
Letter Match Accuracy: 73.00%
Letter Match Standard Deviation: 0.481
Data Source: jama
  Correct Predictions: 458
  Total Predictions: 582
  Accuracy: 78.69%
  Std Dev: 0.4098

Data Source: medxpert
  Correct Predictions: 131
  Total Predictions: 318
  Accuracy: 41.19%
  Std Dev: 0.4930

Data Source: medbullets
  Correct Predictions: 174
  Total Predictions: 207
  Accuracy: 84.06%
  Std Dev: 0.3670

Data Source: mmlu
  Correct Predictions: 186
  Total Predictions: 193
  Accuracy: 96.37%
  Std Dev: 0.1874

